In [1]:
!pip install -q transformers datasets evaluate rouge_score accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch

In [3]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [4]:
from datasets import load_dataset

train_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0", split="train[:40000]")
val_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0", split="validation[:2000]")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [6]:
print(train_dataset)
print(val_dataset)
print(train_dataset[0])

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 40000
})
Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 2000
})
{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able 

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [8]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
  inputs = examples["article"]
  targets = examples["highlights"]
  model_inputs = tokenizer(
      inputs, max_length=max_input_length, truncate=True
  )
  labels = tokenizer(
      targets, max_length=max_target_length, truncate=True
  )
  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

print(tokenized_train)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 40000
})


In [9]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
  decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  result = rouge.compute(
      predictions=decoded_preds,
      references=decoded_labels,
      use_stemmer=True
  )
  return {k: round(v, 4) for k, v in result.items()}

In [10]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-summarizer-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=4,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,8.413674,2.010991,0.358000,0.145000,0.245800,0.331200
2,7.764407,2.017426,0.357700,0.145600,0.244800,0.330900
3,7.622436,2.013063,0.357600,0.145900,0.245700,0.330500
4,7.275162,2.014140,0.357900,0.147100,0.246700,0.331100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=7.944566738891601, metrics={'train_runtime': 7131.7894, 'train_samples_per_second': 22.435, 'train_steps_per_second': 1.402, 'total_flos': 4.872928268242944e+16, 'train_loss': 7.944566738891601, 'epoch': 4.0})

In [12]:
final_metrics = trainer.evaluate()
print(final_metrics)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
7.275162,2.014140,4,0.357900,0.147100,0.246700,0.331100


{'eval_loss': 2.0141403675079346, 'eval_rouge1': 0.3579, 'eval_rouge2': 0.1471, 'eval_rougeL': 0.2467, 'eval_rougeLsum': 0.3311}


In [ ]:
save_dir = f"./{model_checkpoint.split('/')[-1]}-40000-final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

In [14]:
zip_name = f"{model_checkpoint.split('/')[-1]}-40000-final.zip"
!zip -r {zip_name} {save_dir}
print(f"Saved and zipped to {zip_name}")

  adding: bart-base-40000-final/ (stored 0%)
  adding: bart-base-40000-final/generation_config.json (deflated 60%)
  adding: bart-base-40000-final/model.safetensors (deflated 8%)
  adding: bart-base-40000-final/tokenizer_config.json (deflated 50%)
  adding: bart-base-40000-final/config.json (deflated 65%)
  adding: bart-base-40000-final/training_args.bin (deflated 53%)
  adding: bart-base-40000-final/tokenizer.json (deflated 82%)
Saved and zipped to bart-base-40000-final.zip
